## Cell 0 — Colab Drive Mount

In [ ]:
import os
USE_COLAB_MOUNT = True
if USE_COLAB_MOUNT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Colab mount skipped: {e}")

## Cell 1 — ⚙️ Ablation Config

> **Change `ABLATION_MODE` here, then Run All.**
> - `"s1s2"` → Sentinel-1 + Sentinel-2 (12 features, should match baseline)
> - `"s1"` → Sentinel-1 only (3 features)
> - `"s2"` → Sentinel-2 only (9 features)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║   ▼▼  CHANGE THIS TO SWITCH ABLATION MODE  ▼▼       ║
ABLATION_MODE = "s1s2"    # "s1s2" | "s1" | "s2"
# ╚══════════════════════════════════════════════════════╝

from pathlib import Path
DATASET_ROOT   = "/content/drive/MyDrive/pheno_crop_v2"
LOCAL_FALLBACK = Path("./dataset/pheno_crop_v2")
OUTPUT_DIR     = Path("./models")
OUTPUT_DIR.mkdir(exist_ok=True)

root = Path(DATASET_ROOT)
if not root.exists():
    if LOCAL_FALLBACK.exists():
        root = LOCAL_FALLBACK.resolve()
        print(f"Using local fallback: {root}")
    else:
        raise FileNotFoundError(f"Dataset not found at '{DATASET_ROOT}' or '{LOCAL_FALLBACK}'.")

s1_dir  = root / "sentinel_1"
s2_dir  = root / "sentinel_2"
gt_path = root / "ground_truth.csv"
print(f"gt exists: {gt_path.exists()}")

## Cell 2 — Imports, Seeds & Feature Selection

In [ ]:
import math, random, warnings, json
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

S1_COLS  = ["VV", "VH", "VH_VV_Ratio"]
S2_COLS  = ["NDVI", "NDWI", "NDRE", "EVI", "SAVI", "MSAVI", "NDMI", "GNDVI", "cloud_pct"]
ALL_COLS = S1_COLS + S2_COLS

if   ABLATION_MODE == "s1": FEAT_COLS = S1_COLS
elif ABLATION_MODE == "s2": FEAT_COLS = S2_COLS
else:                       FEAT_COLS = ALL_COLS

IN_FEATURES = len(FEAT_COLS)
LOOKBACK = 90
MAX_T    = 30

print(f"{'='*60}")
print(f"  ABLATION MODE : {ABLATION_MODE.upper()}")
print(f"  Features ({IN_FEATURES}): {FEAT_COLS}")
print(f"  CUDA          : {torch.cuda.is_available()}")
print(f"{'='*60}")

## Cell 3 — Labels & Plot-Level Split

Identical `GroupShuffleSplit(SEED=42, 70/15/15)` as model_3 / model_4 for fair comparison.

In [ ]:
gt = pd.read_csv(gt_path)
gt["date"] = pd.to_datetime(gt["date"])

stage_names  = sorted(gt["stage_name"].dropna().unique())
stage_to_idx = {n: i for i, n in enumerate(stage_names)}
idx_to_stage = {i: n for n, i in stage_to_idx.items()}
gt["stage_idx"] = gt["stage_name"].map(stage_to_idx)
num_classes = len(stage_names)
print(f"Classes ({num_classes}): {stage_to_idx}")

plot_df = pd.DataFrame({"plot_id": sorted(gt["plot_id"].unique())})

gss = GroupShuffleSplit(1, test_size=0.15, random_state=SEED)
tv_idx, te_idx = next(gss.split(plot_df, groups=plot_df["plot_id"]))
tv_df      = plot_df.iloc[tv_idx].reset_index(drop=True)
test_plots = set(plot_df.iloc[te_idx]["plot_id"])

gss2 = GroupShuffleSplit(1, test_size=0.1765, random_state=SEED)
tr_idx, va_idx = next(gss2.split(tv_df, groups=tv_df["plot_id"]))
train_plots = set(tv_df.iloc[tr_idx]["plot_id"])
val_plots   = set(tv_df.iloc[va_idx]["plot_id"])

assert train_plots.isdisjoint(val_plots) and train_plots.isdisjoint(test_plots)

train_rows = gt[gt["plot_id"].isin(train_plots)]
val_rows   = gt[gt["plot_id"].isin(val_plots)]
test_rows  = gt[gt["plot_id"].isin(test_plots)]
print(f"Plots  train={len(train_plots)} val={len(val_plots)} test={len(test_plots)}")
print(f"Rows   train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")

## Cell 4 — Ablation-Aware Dataset

Loads only the required modality based on `ABLATION_MODE`. Output shape: `[MAX_T, IN_FEATURES]`.

In [ ]:
class PhenoCropDataset(Dataset):
    """Returns per-timestep features for the selected ABLATION_MODE."""
    def __init__(self, gt_df, s1_dir, s2_dir):
        self.gt = gt_df.sort_values(["plot_id", "date"]).reset_index(drop=True)
        self.s1_cache, self.s2_cache = {}, {}
        for pid in sorted(self.gt["plot_id"].unique()):
            p1 = Path(s1_dir) / f"plot_{pid}_sar.csv"
            p2 = Path(s2_dir) / f"plot_{pid}_indices.csv"
            if p1.exists() and ABLATION_MODE in ("s1", "s1s2"):
                d = pd.read_csv(p1); d["date"] = pd.to_datetime(d["date"])
                self.s1_cache[pid] = d
            if p2.exists() and ABLATION_MODE in ("s2", "s1s2"):
                d = pd.read_csv(p2); d["date"] = pd.to_datetime(d["date"])
                self.s2_cache[pid] = d

    def __len__(self): return len(self.gt)

    def _window(self, df, target_date, cols):
        feats = np.zeros((MAX_T, len(cols)), dtype=np.float32)
        valid = np.zeros(MAX_T, dtype=bool)
        if df is None or df.empty: return feats, valid
        for c in cols:
            if c not in df.columns: df[c] = 0.0
        start = target_date - pd.Timedelta(days=LOOKBACK)
        win = df[(df["date"] > start) & (df["date"] <= target_date)].sort_values("date")
        if win.empty: return feats, valid
        n = min(len(win), MAX_T)
        feats[-n:] = win[cols].fillna(0.0).to_numpy(dtype=np.float32)[-n:]
        valid[-n:] = True
        return feats, valid

    def __getitem__(self, idx):
        row   = self.gt.iloc[idx]
        pid, tdate, label = int(row["plot_id"]), row["date"], int(row["stage_idx"])
        if ABLATION_MODE == "s1":
            feats, valid = self._window(self.s1_cache.get(pid), tdate, S1_COLS)
        elif ABLATION_MODE == "s2":
            feats, valid = self._window(self.s2_cache.get(pid), tdate, S2_COLS)
        else:
            s1f, s1v = self._window(self.s1_cache.get(pid), tdate, S1_COLS)
            s2f, s2v = self._window(self.s2_cache.get(pid), tdate, S2_COLS)
            feats, valid = np.concatenate([s1f, s2f], axis=-1), s1v | s2v
        return {
            "feats": torch.tensor(feats),
            "mask":  torch.tensor(~valid),
            "label": torch.tensor(label).long(),
        }

train_ds = PhenoCropDataset(train_rows, s1_dir, s2_dir)
val_ds   = PhenoCropDataset(val_rows,   s1_dir, s2_dir)
test_ds  = PhenoCropDataset(test_rows,  s1_dir, s2_dir)
print(f"Datasets: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}")
s = train_ds[0]
print(f"  feats: {tuple(s['feats'].shape)}  mask: {tuple(s['mask'].shape)}")

## Cell 5 — Flat Features (Classical ML)

In [ ]:
def extract_flat_features(gt_df):
    """Mean/std/last-pooled flat features for the selected ABLATION_MODE."""
    s1_cache, s2_cache = {}, {}
    for pid in sorted(gt_df["plot_id"].unique()):
        p1 = Path(s1_dir) / f"plot_{pid}_sar.csv"
        p2 = Path(s2_dir) / f"plot_{pid}_indices.csv"
        if p1.exists() and ABLATION_MODE in ("s1", "s1s2"):
            d = pd.read_csv(p1); d["date"] = pd.to_datetime(d["date"])
            s1_cache[pid] = d
        if p2.exists() and ABLATION_MODE in ("s2", "s1s2"):
            d = pd.read_csv(p2); d["date"] = pd.to_datetime(d["date"])
            s2_cache[pid] = d

    if   ABLATION_MODE == "s1": groups = [(s1_cache, S1_COLS)]
    elif ABLATION_MODE == "s2": groups = [(s2_cache, S2_COLS)]
    else:                       groups = [(s1_cache, S1_COLS), (s2_cache, S2_COLS)]

    X, y = [], []
    for _, row in gt_df.iterrows():
        pid, tdate, label = int(row["plot_id"]), row["date"], int(row["stage_idx"])
        row_feats = []
        for cache, cols in groups:
            df = cache.get(pid)
            if df is not None and not df.empty:
                for c in cols:
                    if c not in df.columns: df[c] = 0.0
                start = tdate - pd.Timedelta(days=LOOKBACK)
                win = df[(df["date"] > start) & (df["date"] <= tdate)]
                if not win.empty:
                    vals = win[cols].fillna(0.0).to_numpy(dtype=np.float32)
                    row_feats.extend([vals.mean(0), vals.std(0), vals[-1]])
                else:
                    row_feats.extend([np.zeros(len(cols))] * 3)
            else:
                row_feats.extend([np.zeros(len(cols))] * 3)
        X.append(np.concatenate(row_feats)); y.append(label)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

print("Extracting flat features...")
X_train, y_train = extract_flat_features(train_rows)
X_val,   y_val   = extract_flat_features(val_rows)
X_test,  y_test  = extract_flat_features(test_rows)
print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

## Cell 6 — Shared Helpers (Eval + Training Loop)

In [ ]:
def evaluate_sklearn(name, model, X, y):
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    f1  = f1_score(y, preds, average="macro", zero_division=0)
    print(f"\n{'='*60}\n  {name} [{ABLATION_MODE.upper()}] | Acc: {acc*100:.2f}%  Macro-F1: {f1:.4f}\n{'='*60}")
    print(classification_report(y, preds, target_names=[idx_to_stage[i] for i in sorted(idx_to_stage)], zero_division=0))
    return f1

def evaluate_dl(name, model, loader, device):
    model.eval(); all_true, all_pred = [], []
    with torch.no_grad():
        for batch in loader:
            preds = model(batch["feats"].to(device), batch["mask"].to(device)).argmax(1)
            all_true.extend(batch["label"].numpy()); all_pred.extend(preds.cpu().numpy())
    acc = accuracy_score(all_true, all_pred)
    f1  = f1_score(all_true, all_pred, average="macro", zero_division=0)
    print(f"\n{'='*60}\n  {name} [{ABLATION_MODE.upper()}] | Acc: {acc*100:.2f}%  Macro-F1: {f1:.4f}\n{'='*60}")
    print(classification_report(all_true, all_pred, target_names=[idx_to_stage[i] for i in sorted(idx_to_stage)], zero_division=0))
    return f1

def class_weights_tensor(ds, nc):
    labels = [ds[i]["label"].item() for i in range(len(ds))]
    counts = torch.bincount(torch.tensor(labels), minlength=nc).clamp(min=1).float()
    return (len(labels) / (nc * counts)).clamp(max=10.0)

def train_dl(name, model, train_loader, val_loader, cw, epochs=80, lr=1e-3, patience=12):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n[{name} | {ABLATION_MODE.upper()}] Training on {device}...")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=cw.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "max", factor=0.5, patience=5)
    best_f1, wait, best_state = -1.0, 0, None

    for epoch in range(1, epochs + 1):
        model.train(); tr_true, tr_pred = [], []
        for batch in train_loader:
            feats, mask, labels = batch["feats"].to(device), batch["mask"].to(device), batch["label"].to(device)
            logits = model(feats, mask); loss = criterion(logits, labels)
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            tr_true.extend(labels.cpu().numpy()); tr_pred.extend(logits.argmax(1).detach().cpu().numpy())
        tr_f1 = f1_score(tr_true, tr_pred, average="macro", zero_division=0)

        model.eval(); va_true, va_pred = [], []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(batch["feats"].to(device), batch["mask"].to(device))
                va_true.extend(batch["label"].numpy()); va_pred.extend(logits.argmax(1).cpu().numpy())
        va_f1 = f1_score(va_true, va_pred, average="macro", zero_division=0)
        scheduler.step(va_f1)

        if va_f1 > best_f1:
            best_f1, wait = va_f1, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        print(f"Epoch {epoch:03d} | Train F1: {tr_f1:.4f} | Val F1: {va_f1:.4f} {'*' if wait==0 else ''}")
        if wait >= patience:
            print(f"Early stopping. Best val F1: {best_f1:.4f}"); break

    if best_state: model.load_state_dict(best_state)
    return model, device

# DataLoaders
BATCH_SIZE   = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
cw  = class_weights_tensor(train_ds, num_classes)
print(f"Class weights: {dict(enumerate(cw.numpy().round(3)))}")

results = {}
PREFIX  = f"ablation_{ABLATION_MODE}"

## B1 — Random Forest

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
import joblib

print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=500, max_depth=None, min_samples_split=5,
    min_samples_leaf=2, max_features="sqrt", class_weight="balanced",
    n_jobs=-1, random_state=SEED,
)
rf.fit(X_train_sc, y_train)
print(f"Val Macro-F1: {f1_score(y_val, rf.predict(X_val_sc), average='macro', zero_division=0):.4f}")

results["Random Forest"] = evaluate_sklearn("Random Forest", rf, X_test_sc, y_test)
joblib.dump(rf, OUTPUT_DIR / f"{PREFIX}_rf.pkl")
print(f"Saved: {PREFIX}_rf.pkl")

## B2 — XGBoost

In [ ]:
print("Training XGBoost...")
sample_weights = compute_sample_weight("balanced", y_train)
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric="mlogloss", tree_method="hist",
    device="cuda" if torch.cuda.is_available() else "cpu",
    random_state=SEED, early_stopping_rounds=30, verbosity=0,
)
xgb_model.fit(X_train_sc, y_train, sample_weight=sample_weights,
              eval_set=[(X_val_sc, y_val)], verbose=False)
print(f"Val Macro-F1: {f1_score(y_val, xgb_model.predict(X_val_sc), average='macro', zero_division=0):.4f}")

results["XGBoost"] = evaluate_sklearn("XGBoost", xgb_model, X_test_sc, y_test)
xgb_model.save_model(str(OUTPUT_DIR / f"{PREFIX}_xgb.json"))
print(f"Saved: {PREFIX}_xgb.json")

## B3 — BiLSTM

In [ ]:
class RNNBaseline(nn.Module):
    def __init__(self, in_features, hidden_dim, num_layers, num_classes, rnn_type="LSTM", dropout=0.3):
        super().__init__()
        bidirectional = rnn_type == "BiLSTM"
        cell = nn.LSTM if rnn_type in ("LSTM", "BiLSTM") else nn.GRU
        self.rnn = cell(in_features, hidden_dim, num_layers, batch_first=True,
                        dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.norm = nn.LayerNorm(out_dim); self.drop = nn.Dropout(dropout)
        self.classifier = nn.Sequential(nn.Linear(out_dim, out_dim // 2), nn.GELU(),
                                        nn.Dropout(dropout), nn.Linear(out_dim // 2, num_classes))
    def forward(self, x, mask):
        valid = (~mask).float()
        out, _ = self.rnn(x); out = self.norm(out)
        out = (out * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True).clamp(min=1)
        return self.classifier(self.drop(out))

bilstm = RNNBaseline(IN_FEATURES, 128, 2, num_classes, "BiLSTM", 0.3)
print(f"BiLSTM params: {sum(p.numel() for p in bilstm.parameters()):,}")
bilstm, device = train_dl("BiLSTM", bilstm, train_loader, val_loader, cw)
results["BiLSTM"] = evaluate_dl("BiLSTM", bilstm, test_loader, device)
torch.save(bilstm.state_dict(), OUTPUT_DIR / f"{PREFIX}_bilstm.pt")
print(f"Saved: {PREFIX}_bilstm.pt")

## B4 — TempCNN

In [ ]:
class TempCNN(nn.Module):
    def __init__(self, in_features, num_classes, channels=64, dropout=0.3):
        super().__init__()
        def cb(i, o, k):
            return nn.Sequential(nn.Conv1d(i, o, k, padding=k//2),
                                 nn.BatchNorm1d(o), nn.ReLU(True), nn.Dropout(dropout))
        self.encoder = nn.Sequential(cb(in_features, channels, 5),
                                     cb(channels, channels, 5), cb(channels, channels*2, 3))
        out_ch = channels * 2
        self.classifier = nn.Sequential(nn.Linear(out_ch, out_ch//2), nn.ReLU(True),
                                        nn.Dropout(dropout), nn.Linear(out_ch//2, num_classes))
    def forward(self, x, mask):
        x = x.transpose(1, 2); x = self.encoder(x)
        valid = (~mask).float()
        x = (x * valid.unsqueeze(1)).sum(-1) / valid.sum(-1, keepdim=True).clamp(min=1)
        return self.classifier(x)

tempcnn = TempCNN(IN_FEATURES, num_classes, 64, 0.3)
print(f"TempCNN params: {sum(p.numel() for p in tempcnn.parameters()):,}")
tempcnn, device = train_dl("TempCNN", tempcnn, train_loader, val_loader, cw)
results["TempCNN"] = evaluate_dl("TempCNN", tempcnn, test_loader, device)
torch.save(tempcnn.state_dict(), OUTPUT_DIR / f"{PREFIX}_tempcnn.pt")
print(f"Saved: {PREFIX}_tempcnn.pt")

## B5 — TCN (Temporal Convolutional Network)

In [ ]:
class _TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch,  out_ch, kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation)
        self.bn1, self.bn2 = nn.BatchNorm1d(out_ch), nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.relu, self.pad = nn.ReLU(True), pad
    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)[..., :-self.pad] if self.pad else self.conv1(x)))
        out = self.drop(out)
        out = self.relu(self.bn2(self.conv2(out)[..., :-self.pad] if self.pad else self.conv2(out)))
        return self.relu(self.drop(out) + res)

class TCN(nn.Module):
    def __init__(self, in_features, num_classes, num_channels=None, kernel_size=3, dropout=0.2):
        super().__init__()
        if num_channels is None: num_channels = [64, 64, 128, 128]
        layers, in_ch = [], in_features
        for i, out_ch in enumerate(num_channels):
            layers.append(_TCNBlock(in_ch, out_ch, kernel_size, 2**i, dropout)); in_ch = out_ch
        self.network = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Linear(in_ch, in_ch//2), nn.ReLU(True),
                                        nn.Dropout(dropout), nn.Linear(in_ch//2, num_classes))
    def forward(self, x, mask):
        x = x.transpose(1, 2); x = self.network(x)
        x = x[..., :mask.shape[1]]
        valid = (~mask).float()
        x = (x * valid.unsqueeze(1)).sum(-1) / valid.sum(-1, keepdim=True).clamp(min=1)
        return self.classifier(x)

tcn = TCN(IN_FEATURES, num_classes, [64, 64, 128, 128], 3, 0.2)
print(f"TCN params: {sum(p.numel() for p in tcn.parameters()):,}")
tcn, device = train_dl("TCN", tcn, train_loader, val_loader, cw)
results["TCN"] = evaluate_dl("TCN", tcn, test_loader, device)
torch.save(tcn.state_dict(), OUTPUT_DIR / f"{PREFIX}_tcn.pt")
print(f"Saved: {PREFIX}_tcn.pt")

## B6 — Vanilla Transformer

In [ ]:
def sinusoid_table(n_pos, d_hid):
    pos = torch.arange(n_pos).float().unsqueeze(1)
    dim = torch.arange(d_hid).float().unsqueeze(0)
    a   = pos / torch.pow(10000, 2 * (dim // 2) / d_hid)
    a[:, 0::2] = torch.sin(a[:, 0::2]); a[:, 1::2] = torch.cos(a[:, 1::2])
    return a

class VanillaTransformer(nn.Module):
    def __init__(self, in_features, d_model, nhead, num_layers, num_classes,
                 dim_feedforward=256, dropout=0.1, max_len=128):
        super().__init__()
        self.input_proj = nn.Linear(in_features, d_model)
        self.register_buffer("pos_enc", sinusoid_table(max_len, d_model))
        enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout,
                                         batch_first=True, norm_first=True)
        self.encoder    = nn.TransformerEncoder(enc, num_layers, nn.LayerNorm(d_model))
        self.classifier = nn.Sequential(nn.Linear(d_model, d_model//2), nn.GELU(),
                                        nn.Dropout(dropout), nn.Linear(d_model//2, num_classes))
    def forward(self, x, mask):
        B, T, _ = x.shape
        x = self.input_proj(x) + self.pos_enc[:T].unsqueeze(0)
        kpm = mask.clone()
        if kpm.all(dim=1).any(): kpm[kpm.all(dim=1), -1] = False
        x = self.encoder(x, src_key_padding_mask=kpm)
        valid = (~kpm).float().unsqueeze(-1)
        x = (x * valid).sum(1) / valid.sum(1).clamp(min=1)
        return self.classifier(x)

vt = VanillaTransformer(IN_FEATURES, 128, 8, 3, num_classes, 256, 0.1, MAX_T + 4)
print(f"Vanilla Transformer params: {sum(p.numel() for p in vt.parameters()):,}")
vt, device = train_dl("Vanilla Transformer", vt, train_loader, val_loader, cw, lr=5e-4)
results["Vanilla Transformer"] = evaluate_dl("Vanilla Transformer", vt, test_loader, device)
torch.save(vt.state_dict(), OUTPUT_DIR / f"{PREFIX}_vanilla_transformer.pt")
print(f"Saved: {PREFIX}_vanilla_transformer.pt")

## Save Results

In [ ]:
out_path = OUTPUT_DIR / f"ablation_{ABLATION_MODE}_results.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved → {out_path}")

print(f"\n{'='*55}\n  [{ABLATION_MODE.upper()}] Summary\n{'='*55}")
for name, f1 in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:<28} {f1:.4f}")
print("="*55)

## Final Ablation Table

Run this cell after completing **all 3 modes** (`s1s2`, `s1`, `s2`).
It loads the saved JSON files and prints the full comparison table.

In [ ]:
import json
from pathlib import Path

modes       = ["s1s2", "s1", "s2"]
col_labels  = ["S1+S2 (baseline)", "S1-only", "S2-only"]
model_order = ["Random Forest", "XGBoost", "BiLSTM", "TempCNN", "TCN", "Vanilla Transformer"]

all_results = {}
for m in modes:
    p = OUTPUT_DIR / f"ablation_{m}_results.json"
    if p.exists():
        with open(p) as f:
            all_results[m] = json.load(f)
    else:
        print(f"[WARNING] {p} not found — run with ABLATION_MODE='{m}' first.")

if len(all_results) == 0:
    print("No results found yet. Run the full notebook for each ABLATION_MODE first.")
else:
    W = 18
    header = f"  {'Model':<28}" + "".join(f"{lbl:>{W}}" for lbl in col_labels)
    divider = "=" * (28 + W * len(modes) + 2)
    print(f"\n{divider}")
    print("  ABLATION STUDY — Test Macro-F1  (delta vs S1+S2)")
    print(divider)
    print(header)
    print("-" * (28 + W * len(modes) + 2))
    base = all_results.get("s1s2", {})
    for name in model_order:
        row = f"  {name:<28}"
        for i, m in enumerate(modes):
            val = all_results.get(m, {}).get(name)
            if val is None:
                row += f"{'—':>{W}}"
            elif i == 0:
                row += f"{val:>{W}.4f}"
            else:
                delta = val - base.get(name, val)
                cell  = f"{val:.4f} ({delta:+.3f})"
                row  += f"{cell:>{W}}"
        print(row)
    print(divider)
    missing = [m for m in modes if m not in all_results]
    if missing:
        print(f"  ⚠ Still needed: {missing} — re-run with ABLATION_MODE set accordingly.")
    else:
        print("  ✓ All 3 modes complete.")
    print(divider)